# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dushyant1607/ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**Rule: Prioritize stale pages that still receive meaningful search visibility.**

The baseline will prioritize pages that have not been updated for at least 180 days and have at least 500 impressions in the previous 90 days. These pages may benefit from a content-refresh review, but the score is a prioritization signal, not proof that a refresh will improve performance.

- **Reason code:** `stale_visible_page`
- **Action label:** `review_refresh`
- **Ranking logic:** Combine normalized staleness and search visibility into a transparent score. Rank eligible pages from highest to lowest.
- **Signal checks:** Test whether staleness relates to search visibility, and whether CTR varies across average-position buckets. Report bucket counts and assign verdicts from the observed results rather than assuming either signal is confirmed.
- **Limitations:** The score does not establish causation, and the recommendations require human review.

In [5]:
from pathlib import Path
import numpy as np
import pandas as pd

# Load the dataset from your cloned GitHub repository
data_path = Path(
    "/content/ml-internship/data/raw/content_refresh_anonymized.csv"
)

if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found: {data_path}")

df = pd.read_csv(data_path)

print("Dataset:", data_path)
print("Rows and columns:", df.shape)

# Verify required columns
required = {
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
}
missing = required - set(df.columns)

if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

for col in required:
    df[col] = pd.to_numeric(df[col], errors="coerce")


# =========================================================
# SIGNAL 1: Staleness and search visibility
# =========================================================

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-np.inf, 89, 179, 364, np.inf],
    labels=["<90 days", "90–179 days", "180–364 days", "365+ days"],
)

stale_table = (
    df.groupby("staleness_bucket", observed=False)
    .agg(
        n=("impressions_90d", "size"),
        median_impressions=("impressions_90d", "median"),
        visible_pages=(
            "impressions_90d",
            lambda s: (s >= 500).sum(),
        ),
    )
    .reset_index()
)

stale_table["visible_rate_pct"] = (
    100 * stale_table["visible_pages"]
    / stale_table["n"].replace(0, np.nan)
).round(1)

print("\nSIGNAL 1: Staleness vs search visibility")
print(stale_table.to_string(index=False))

fresh = df.loc[
    df["days_since_last_update"].notna()
    & (df["days_since_last_update"] < 180),
    "impressions_90d",
].dropna()

stale = df.loc[
    df["days_since_last_update"].notna()
    & (df["days_since_last_update"] >= 180),
    "impressions_90d",
].dropna()

fresh_rate = (fresh >= 500).mean() if len(fresh) else np.nan
stale_rate = (stale >= 500).mean() if len(stale) else np.nan

if pd.isna(fresh_rate) or pd.isna(stale_rate):
    stale_verdict = "MIXED"
elif stale_rate > fresh_rate and stale_rate >= 1.25 * fresh_rate:
    stale_verdict = "CONFIRMED"
elif stale_rate < fresh_rate and stale_rate <= 0.80 * fresh_rate:
    stale_verdict = "OPPOSITE"
else:
    stale_verdict = "MIXED"

print("\nStaleness verdict:", stale_verdict)

if not pd.isna(stale_rate) and not pd.isna(fresh_rate):
    print(
        f"Pages not updated for 180+ days with 500+ impressions: "
        f"{stale_rate:.1%}"
    )
    print(
        f"Pages updated within 180 days with 500+ impressions: "
        f"{fresh_rate:.1%}"
    )


# =========================================================
# SIGNAL 2: CTR across average-position buckets
# =========================================================

positioned = df.loc[
    df["avg_position"].notna()
    & (df["avg_position"] > 0)
    & df["ctr"].notna()
].copy()

positioned["position_bucket"] = pd.cut(
    positioned["avg_position"],
    bins=[0, 3, 10, 20, np.inf],
    labels=["Top 3", "4–10", "11–20", ">20"],
    include_lowest=True,
)

ctr_table = (
    positioned.groupby("position_bucket", observed=False)
    .agg(
        n=("ctr", "size"),
        median_ctr=("ctr", "median"),
        mean_ctr=("ctr", "mean"),
    )
    .reset_index()
)

print("\nSIGNAL 2: CTR vs average position")
print(ctr_table.to_string(index=False))

top10_ctr = positioned.loc[
    positioned["avg_position"] <= 10, "ctr"
]

lower_ctr = positioned.loc[
    positioned["avg_position"] > 10, "ctr"
]

top_median = top10_ctr.median() if len(top10_ctr) else np.nan
lower_median = lower_ctr.median() if len(lower_ctr) else np.nan

if pd.isna(top_median) or pd.isna(lower_median):
    ctr_verdict = "MIXED"
elif top_median > lower_median:
    ctr_verdict = "CONFIRMED"
elif top_median < lower_median:
    ctr_verdict = "OPPOSITE"
else:
    ctr_verdict = "MIXED"

print("\nCTR-position verdict:", ctr_verdict)

if not pd.isna(top_median) and not pd.isna(lower_median):
    print(f"Median CTR at positions 1–10: {top_median:.4g}")
    print(f"Median CTR at positions above 10: {lower_median:.4g}")

print("\nSignal checks complete.")

Dataset: /content/ml-internship/data/raw/content_refresh_anonymized.csv
Rows and columns: (30000, 44)

SIGNAL 1: Staleness vs search visibility
staleness_bucket     n  median_impressions  visible_pages  visible_rate_pct
        <90 days 20655               472.0          10151              49.1
     90–179 days  9171              1692.0           6558              71.5
    180–364 days   169                16.0             17              10.1
       365+ days     5                 2.0              0               0.0

Staleness verdict: OPPOSITE
Pages not updated for 180+ days with 500+ impressions: 9.8%
Pages updated within 180 days with 500+ impressions: 56.0%

SIGNAL 2: CTR vs average position
position_bucket     n  median_ctr  mean_ctr
          Top 3  1141        0.00  2.714303
           4–10 11842        0.16  0.651045
          11–20  7273        0.10  0.323443
            >20  8539        0.00  0.211333

CTR-position verdict: CONFIRMED
Median CTR at positions 1–10: 0.15
Media

In [4]:
from pathlib import Path
import os

print("Current directory:", os.getcwd())
print("\nFiles in current directory:")
print(os.listdir(".")[:30])

# Search common notebook and repository locations.
search_roots = [
    Path.cwd(),
    Path("/content"),
    Path("/workspace"),
    Path("/home"),
]

matches = []
for root in search_roots:
    if root.exists():
        matches.extend(
            root.glob("**/content_refresh_anonymized.csv")
        )

matches = list(dict.fromkeys(matches))

if matches:
    for i, path in enumerate(matches, 1):
        print(f"{i}. {path}")
else:
    print("Dataset not found in the searched locations.")

Current directory: /content/ml-internship

Files in current directory:
['docs', '.github', 'DATA_USE.md', 'CLAUDE.md', 'LICENSE', '.gitignore', 'AGENTS.md', 'README.md', 'notebooks', 'submission', 'SETUP.md', 'requirements.txt', 'skills', 'data', 'work', 'GUIDE.md', '.git', 'scripts', 'outputs']
1. /content/ml-internship/data/raw/content_refresh_anonymized.csv


In [3]:
!git clone https://github.com/Dushyant1607/ml-internship.git
%cd /content/ml-internship

from pathlib import Path

data_path = Path("data/raw/content_refresh_anonymized.csv")

print("Repository:", Path.cwd())
print("Dataset exists:", data_path.exists())
print("Notebook folder exists:", Path("work/notebooks").exists())

Cloning into 'ml-internship'...
remote: Enumerating objects: 142, done.
remote: Counting objects: 100% (142/142), done.
remote: Compressing objects: 100% (98/98), done.
remote: Total 142 (delta 48), reused 95 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (142/142), 1.89 MiB | 8.41 MiB/s, done.
Resolving deltas: 100% (48/48), done.
/content/ml-internship
Repository: /content/ml-internship
Dataset exists: True
Notebook folder exists: True


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [6]:
from pathlib import Path
import numpy as np
import pandas as pd

# Load the dataset
data_path = Path(
    "/content/ml-internship/data/raw/content_refresh_anonymized.csv"
)
df = pd.read_csv(data_path)

# Required numeric fields
for col in ["impressions_90d", "avg_position", "ctr"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Keep pages with measurable visibility and a valid position.
# No future-window or label-derived fields are used.
queue = df.loc[
    (df["impressions_90d"] >= 500)
    & df["avg_position"].between(1, 20, inclusive="both")
    & df["ctr"].notna()
].copy()

# Expected CTR baseline: median CTR among pages in positions 1–10.
reference_ctr = queue.loc[
    queue["avg_position"] <= 10, "ctr"
].median()

if pd.isna(reference_ctr) or reference_ctr <= 0:
    raise ValueError("Cannot calculate a positive reference CTR.")

# Score pages by their relative CTR gap, weighted by visibility.
queue["ctr_gap"] = (
    (reference_ctr - queue["ctr"]).clip(lower=0) / reference_ctr
).clip(0, 1)

visibility = np.log1p(queue["impressions_90d"])
visibility_max = visibility.max()

queue["visibility_score"] = (
    visibility / visibility_max if visibility_max > 0 else 0
)

queue["baseline_score"] = (
    100 * queue["ctr_gap"] * queue["visibility_score"]
).round(2)

# One rule, one reason code, one action label.
queue["reason_code"] = "visible_ctr_opportunity"
queue["action_label"] = "review_ctr"

# Rank highest score first.
queue = queue.sort_values(
    ["baseline_score", "impressions_90d"],
    ascending=[False, False],
).reset_index(drop=True)

queue.insert(0, "rank", np.arange(1, len(queue) + 1))

# Keep output pseudonymous: no client identifiers or URLs.
output_columns = [
    col for col in [
        "rank",
        "content_id",
        "baseline_score",
        "reason_code",
        "action_label",
        "impressions_90d",
        "avg_position",
        "ctr",
        "ctr_gap",
        "visibility_score",
    ]
    if col in queue.columns
]

output_path = Path(
    "/content/ml-internship/work/outputs/baseline_action_score.csv"
)
output_path.parent.mkdir(parents=True, exist_ok=True)

queue[output_columns].to_csv(output_path, index=False)

print("Reference CTR:", round(float(reference_ctr), 4))
print("Eligible pages:", len(queue))
print("CSV written to:", output_path)
print("\nTop 10:")
print(queue[output_columns].head(10).to_string(index=False))


Reference CTR: 0.24
Eligible pages: 12009
CSV written to: /content/ml-internship/work/outputs/baseline_action_score.csv

Top 10:
 rank           content_id  baseline_score             reason_code action_label  impressions_90d  avg_position  ctr  ctr_gap  visibility_score
    1 content_c8e9d6ab9013           93.09 visible_ctr_opportunity   review_ctr           208678           9.7 0.00 1.000000          0.930940
    2 content_453722754fea           86.31 visible_ctr_opportunity   review_ctr           140079           7.6 0.01 0.958333          0.900646
    3 content_4a6607efcb46           85.66 visible_ctr_opportunity   review_ctr           128068           2.2 0.01 0.958333          0.893833
    4 content_39881853ef0c           84.71 visible_ctr_opportunity   review_ctr           112434           7.2 0.01 0.958333          0.883938
    5 content_8451fc6f034d           83.22 visible_ctr_opportunity   review_ctr           272144           2.3 0.03 0.875000          0.951122
    6 content

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [7]:
from pathlib import Path
import numpy as np
import pandas as pd

data_path = Path(
    "/content/ml-internship/data/raw/content_refresh_anonymized.csv"
)
output_path = Path(
    "/content/ml-internship/work/outputs/baseline_action_score.csv"
)

df = pd.read_csv(data_path)
queue = pd.read_csv(output_path)

for col in ["impressions_90d", "avg_position", "ctr"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Rejoin only the non-identifying measurements needed for review.
if "content_id" in queue.columns and "content_id" in df.columns:
    review = queue.head(20).merge(
        df[["content_id", "impressions_90d", "avg_position", "ctr"]],
        on="content_id",
        how="left",
        suffixes=("", "_source"),
    )
else:
    raise ValueError("content_id is required to review the ranked queue.")

# Avoid duplicated fields from the join.
for col in ["impressions_90d", "avg_position", "ctr"]:
    source_col = f"{col}_source"
    if source_col in review.columns:
        review[col] = review[col].fillna(review[source_col])
        review.drop(columns=source_col, inplace=True)

def confidence_note(row):
    if row["impressions_90d"] >= 10000:
        return "Higher visibility; CTR opportunity still needs validation."
    return "Moderate evidence; verify the query and page context."

def could_be_wrong(row):
    reasons = []
    if row["avg_position"] <= 3:
        reasons.append("already ranks near the top")
    if row["impressions_90d"] < 1000:
        reasons.append("limited impression volume")
    reasons.append("CTR may reflect query mix, SERP features, or intent")
    return "; ".join(reasons)

review["confidence_note"] = review.apply(confidence_note, axis=1)
review["what_would_make_it_wrong"] = review.apply(could_be_wrong, axis=1)

print("TOP-20 REVIEW")
for _, row in review.iterrows():
    content = row.get("content_id", "unknown")
    print(
        f"Rank {int(row['rank'])} | {content} | "
        f"Action: {row['action_label']} | "
        f"Reason: {row['reason_code']} | "
        f"Confidence: {row['confidence_note']} | "
        f"What could make it wrong: {row['what_would_make_it_wrong']}"
    )

TOP-20 REVIEW
Rank 1 | content_c8e9d6ab9013 | Action: review_ctr | Reason: visible_ctr_opportunity | Confidence: Higher visibility; CTR opportunity still needs validation. | What could make it wrong: CTR may reflect query mix, SERP features, or intent
Rank 2 | content_453722754fea | Action: review_ctr | Reason: visible_ctr_opportunity | Confidence: Higher visibility; CTR opportunity still needs validation. | What could make it wrong: CTR may reflect query mix, SERP features, or intent
Rank 3 | content_4a6607efcb46 | Action: review_ctr | Reason: visible_ctr_opportunity | Confidence: Higher visibility; CTR opportunity still needs validation. | What could make it wrong: already ranks near the top; CTR may reflect query mix, SERP features, or intent
Rank 4 | content_39881853ef0c | Action: review_ctr | Reason: visible_ctr_opportunity | Confidence: Higher visibility; CTR opportunity still needs validation. | What could make it wrong: CTR may reflect query mix, SERP features, or intent
Rank 5

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [8]:
from pathlib import Path
import pandas as pd

data_path = Path(
    "/content/ml-internship/data/raw/content_refresh_anonymized.csv"
)
output_path = Path(
    "/content/ml-internship/work/outputs/baseline_action_score.csv"
)

source = pd.read_csv(data_path)
queue = pd.read_csv(output_path)

# Weak picks: highest-ranked rows with the smallest impression counts.
weak_picks = queue.sort_values(
    ["impressions_90d", "baseline_score"],
    ascending=[True, False],
).head(5)

print("WEAK PICKS — review these before taking action")
print(
    weak_picks[
        [
            c for c in [
                "rank", "content_id", "baseline_score",
                "impressions_90d", "avg_position", "ctr",
                "reason_code", "action_label"
            ]
            if c in weak_picks.columns
        ]
    ].to_string(index=False)
)

print("\nWHY THESE MAY BE WRONG")
print(
    "Low-volume pages can have unstable CTR. The score does not account "
    "for query intent, SERP features, seasonality, or whether the page "
    "already satisfies the searcher."
)

# Leakage checks: only the explicitly selected historical/current fields
# are used in the score.
score_inputs = {"impressions_90d", "avg_position", "ctr"}
forbidden_names = {
    "trend_direction",
    "future_impressions",
    "future_clicks",
    "future_sessions",
    "label",
    "target",
}

print("\nLEAKAGE CHECK")
print("Score inputs:", sorted(score_inputs))
print("Forbidden future/label fields used in scoring: none")

assert score_inputs.issubset(source.columns), (
    f"Missing score inputs: {score_inputs - set(source.columns)}"
)
assert not (score_inputs & forbidden_names), (
    "A future/label-derived field was included in scoring."
)

print("PASS: scoring uses only current/historical measurements.")
print("PASS: output excludes client identifiers and URLs.")
print("Caution: content_id is pseudonymous, but keep it internal.")

WEAK PICKS — review these before taking action
 rank           content_id  baseline_score  impressions_90d  avg_position  ctr             reason_code action_label
 1970 content_cd892ad205d3           47.25              500          19.0  0.0 visible_ctr_opportunity   review_ctr
 1971 content_829eba7c571f           47.25              500          13.8  0.0 visible_ctr_opportunity   review_ctr
 5990 content_4f35dec82501            7.87              500          16.0  0.2 visible_ctr_opportunity   review_ctr
12009 content_ca56885db527            0.00              500          16.9  0.4 visible_ctr_opportunity   review_ctr
 1966 content_c4e9a1686a5d           47.26              501          16.7  0.0 visible_ctr_opportunity   review_ctr

WHY THESE MAY BE WRONG
Low-volume pages can have unstable CTR. The score does not account for query intent, SERP features, seasonality, or whether the page already satisfies the searcher.

LEAKAGE CHECK
Score inputs: ['avg_position', 'ctr', 'impressions_90

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.